# 🚀 VÒNG LẶP TỰ GÁN NHÃN THỨ HAI (SELF-TRAINING / PSEUDO-LABELING)
### Huấn luyện OCR Nôm trên 51.371 ảnh GOLD và Giải cứu 12.871 ô REVIEW

Notebook này chạy trên **Kaggle GPU (T4 x2 hoặc P100)**.

**Mục tiêu bài toán:**
1. Huấn luyện mô hình CNN nhận diện 1.331 lớp ký tự chữ Nôm viết tay bút lông từ chính 51k ảnh GOLD vừa sinh ra.
2. Khắc phục điểm nghẽn của OCR đầu vào (S1): dùng mô hình vừa học nét bút lông để suy diễn lại 12.871 ô đang kẹt ở REVIEW.
3. Giao thoa với từ điển âm-chữ để tự động thăng cấp các ô có độ tin cậy cao lên **GOLD v2**.

**Cài đặt:** Ở panel bên phải -> **Settings -> Accelerator -> GPU T4 x2 (hoặc GPU P100)**.

In [ ]:
!nvidia-smi

### 1. Tìm tệp dữ liệu đầu vào (`crops.npz`, `labels_final.csv`, `QuocNgu_SinoNom.csv`)

In [ ]:
import os, sys, glob
from pathlib import Path

crops_files = glob.glob("/kaggle/input/**/crops.npz", recursive=True) + glob.glob("**/crops.npz", recursive=True)
labels_files = glob.glob("/kaggle/input/**/labels_final.csv", recursive=True) + glob.glob("**/labels_final.csv", recursive=True)
dict_files = glob.glob("/kaggle/input/**/QuocNgu_SinoNom.csv", recursive=True) + glob.glob("**/QuocNgu_SinoNom.csv", recursive=True)
script_files = glob.glob("/kaggle/input/**/train_self_training_ocr.py", recursive=True) + glob.glob("**/train_self_training_ocr.py", recursive=True)

assert len(crops_files) > 0, "Không tìm thấy crops.npz! Hãy kiểm tra đã Add Dataset vào notebook chưa."
assert len(labels_files) > 0, "Không tìm thấy labels_final.csv!"
assert len(dict_files) > 0, "Không tìm thấy QuocNgu_SinoNom.csv!"

crops_path = crops_files[0]
labels_path = labels_files[0]
dict_path = dict_files[0]
script_path = script_files[0] if script_files else "train_self_training_ocr.py"

print(f"✓ crops.npz:          {crops_path} ({os.path.getsize(crops_path) / (1024*1024):.2f} MB)")
print(f"✓ labels_final.csv:   {labels_path} ({os.path.getsize(labels_path) / (1024*1024):.2f} MB)")
print(f"✓ QuocNgu_SinoNom.csv:{dict_path} ({os.path.getsize(dict_path) / (1024*1024):.2f} MB)")
print(f"✓ train script:       {script_path}")

### 2. Chạy Huấn luyện OCR Nôm & Suy diễn Giải cứu REVIEW
Thời gian chạy ước tính trên GPU T4 / P100: **~6–8 phút** (15 epochs).

In [ ]:
# Đảm bảo tương thích độ dài giữa crops.npz và labels_final.csv nếu lệch thế hệ
with open(script_path) as f:
    src_code = f.read()
if 'assert len(X_all) == len(df_labels)' in src_code:
    src_code = src_code.replace('assert len(X_all) == len(df_labels), "Số lượng ảnh và dòng nhãn không khớp!"',
                                'min_len = min(len(X_all), len(df_labels))\n    X_all = X_all[:min_len]\n    df_labels = df_labels.iloc[:min_len].reset_index(drop=True)')
    with open('/kaggle/working/train_self_training_ocr.py', 'w') as f:
        f.write(src_code)
    script_path = '/kaggle/working/train_self_training_ocr.py'

!python "{script_path}" \
    --crops "{crops_path}" \
    --labels "{labels_path}" \
    --dict "{dict_path}" \
    --out-dir /kaggle/working/output

### 3. Đánh giá & Xem trực quan kết quả giải cứu

In [ ]:
import pandas as pd, json

out_csv = "/kaggle/working/output/review_pseudo_labels.csv"
metrics_json = "/kaggle/working/output/training_metrics.json"

assert os.path.exists(out_csv), "Không thấy file kết quả review_pseudo_labels.csv!"
df_res = pd.read_csv(out_csv)

print(f"✓ review_pseudo_labels.csv có {len(df_res):,} dòng ({os.path.getsize(out_csv) / (1024*1024):.2f} MB).")
print("\n--- PHÂN BỐ KẾT QUẢ GIẢI CỨU ---")
print(df_res["decision"].value_counts())

print("\n--- MẪU 10 Ô ĐƯỢC GIẢI CỨU ĐỘ TIN CẬY CAO NHẤT ---")
rescued = df_res[df_res["decision"] == "RESCUE_GOLD_HIGH"].sort_values("prob", ascending=False)
display(rescued[["book", "page", "column", "syllable", "ocr_char_old", "predicted_nom", "unicode", "prob"]].head(10))

if os.path.exists(metrics_json):
    with open(metrics_json) as f:
        metrics = json.load(f)
    last_ep = metrics[-1]
    print(f"\n✓ Epoch cuối cùng ({last_ep['epoch']}): Val Top-1 = {last_ep['val_top1']:.1%}, Val Top-5 = {last_ep['val_top5']:.1%}")

### 4. Đóng gói kết quả để tải về máy Mac
Tải tệp `/kaggle/working/self_training_results.zip` về và giải nén vào thư mục `lab/self_training_v2/` trên máy Mac.

In [ ]:
!zip -j /kaggle/working/self_training_results.zip \
    /kaggle/working/output/review_pseudo_labels.csv \
    /kaggle/working/output/training_metrics.json \
    /kaggle/working/output/best_nom_ocr.pt

zip_size = os.path.getsize("/kaggle/working/self_training_results.zip") / (1024*1024)
print(f"\n✅ ĐÃ TẠO TỆP KẾT QUẢ: /kaggle/working/self_training_results.zip ({zip_size:.2f} MB)")
print("👉 Ở panel bên phải mục Output -> bấm download 'self_training_results.zip'!")